# simulate_arma

> 对应代码：`EconometricsCode/code_in_notes/Chap.26/simulate_arma.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.26`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.26")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们生成 500 期时间轴与标准正态白噪声 e，作为两个 ARMA(1,1) 过程共同的冲击。

In [ ]:
%%stata
clear
set obs 500
set seed 505
gen t=_n
gen e=rnormal()
tsset t

我们构造两个 ARMA(1,1) 过程，使用相同的冲击 e：y1 为 y_t = 0.5 y_{t-1} + e_t - 0.5 e_{t-1}（AR 系数 φ=0.5，MA 系数 θ=-0.5）；y2 为 y_t = -0.5 y_{t-1} + e_t + 0.9 e_{t-1}（AR 系数为负、MA 系数接近 1）。两个过程均满足平稳性与可逆性条件，但动态形态明显不同。

In [ ]:
%%stata
gen y1=0
replace y1=0.5*L.y1+e-0.5*L.e if t>1
gen y2=0
replace y2=-0.5*L.y2+e+0.9*L.e if t>1

我们把 y1 与原始冲击 e 画在一起对比：ARMA 过程是冲击的平滑加权平均，波动比白噪声更平缓。

In [ ]:
%%stata
tsline y1 e

绘制 y1 的样本 ACF。ARMA(1,1) 的自相关函数从滞后 1 期之后开始以 φ^k 几何衰减，但一阶自相关系数同时受 φ 和 θ 影响，这是 ARMA 与纯 AR 的细微区别。

In [ ]:
%%stata
ac y1

画出 y2 的时间序列图，观察负 AR 系数带来的交替波动与较强 MA 成分的叠加效果。

In [ ]:
%%stata
tsline y2

绘制 y2 的 ACF：一阶之后以 (-0.5)^k 正负交替衰减，帮助我们理解 ARMA(1,1) 自相关结构由 AR 系数主导衰减模式的规律。

In [ ]:
%%stata
ac y2